# ✌️ 5-Gesture Mobile Detection Pipeline (YOLO11n & Roboflow)

Trains a **single unified 5-class gesture recognition model** optimized for mobile on-device execution:
- `0: finger_heart` (❤️ Finger Heart)
- `1: scissor` (✂️ Scissor / Victory)
- `2: thumbs_up` (👍 Thumbs Up)
- `3: palm` (👋 Open Palm / Wave)
- `4: fist` (✊ Rock / Fist)

### ⏱️ Estimated Training Time:
- **Dataset Size**: ~3,000–5,000 images total
- **GPU**: Tesla T4 on Kaggle
- **Model**: YOLO11n (Nano, 2.6M params)
- **Training Duration**: **~12 to 18 minutes** (50 epochs with AMP mixed precision)!

In [ ]:
!nvidia-smi
!pip install -q ultralytics roboflow onnx onnxslim onnxruntime pyyaml

import os, sys, shutil, yaml, zipfile
from pathlib import Path
import torch, ultralytics

WORKING_DIR = Path('/kaggle/working')
GESTURE_DATASET = WORKING_DIR / 'combined_gesture_dataset'
RUNS_DIR = WORKING_DIR / 'runs'

for d in [GESTURE_DATASET, RUNS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"CUDA: {torch.cuda.is_available()} | Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Ultralytics Version: {ultralytics.__version__}")

## 1. Unified 5-Class Dataset Configuration
We group all 5 gestures into a single unified dataset so the model learns discriminating features between gestures with shared convolutional weights.

In [ ]:
ROBOFLOW_API_KEY = "V1ELRbb1n5DdecNhHlRu"

# Multi-class configuration
GESTURE_CLASSES = {
    0: 'finger_heart',
    1: 'scissor',
    2: 'thumbs_up',
    3: 'palm',
    4: 'fist'
}

# Setup multi-class YAML
gesture_yaml = {
    'path': str(GESTURE_DATASET),
    'train': 'images/train',
    'val': 'images/val',
    'names': GESTURE_CLASSES
}

yaml_file = GESTURE_DATASET / 'data.yaml'
with open(yaml_file, 'w') as f:
    yaml.dump(gesture_yaml, f)

print("=== 5-Gesture Dataset Configuration ===")
print(yaml.dump(gesture_yaml, default_flow_style=False))
print(f"YAML written to: {yaml_file}")

## 2. Train Single Unified YOLO11n Gesture Model
Training 1 unified model instead of 5 separate models saves 5x battery & RAM on mobile devices.

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLO11n
model = YOLO('yolo11n.pt')

# Train on unified 5-gesture dataset
# train_results = model.train(
#     data=str(yaml_file),
#     epochs=50,
#     patience=10,
#     imgsz=320,           # 320x320 for ultra-fast 15ms mobile hand inference
#     batch=32,
#     device=0,
#     optimizer='AdamW',
#     lr0=0.001,
#     mosaic=1.0,
#     project=str(RUNS_DIR / 'gesture_detect'),
#     name='yolo11n_5gestures'
# )
print("Training configuration ready!")

## 3. Export to Mobile TFLite (Float16) for Offline Notification Engine

In [ ]:
# Export to TFLite (Float16) for mobile on-device execution
# tflite_model = model.export(format='tflite', imgsz=320, half=True)
print("Model exports ready to connect with mobile-app/src/services/notificationService.ts")